# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [30]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):                              # Try the API request up to the specified number of times
      r = requests.get(BASE, params=params, timeout=10)       # Send the request and stop waiting if it takes longer than 10 seconds
      if r.status_code == 200:                                # If successful, convert the daily weather data into a DataFrame
        return pd.DataFrame(r.json()['daily'])
      if r.status_code not in RETRY_STATUSES:                 # If the error is not temporary, raise an exception
        r.raise_for_status()
      if attempt < (tries - 1):                               # Wait longer before retrying
        time.sleep(2**attempt)
    raise RuntimeError(f"Weather API failed after {tries} attempts")
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [31]:
used_fallback = False

try:
    weather = get_weather('2026-09-05', '2026-10-03')                         # Request weather for the entire date range in one API call
    weather['time'] = pd.to_datetime(weather['time'])                         # Convert the date column from strings to datetime values
    dates = pd.to_datetime(['2026-09-05', '2026-10-03'])                      # Define the two specific dates we want to keep
    weather = weather[weather['time'].isin(dates)].reset_index(drop=True)     # Keep only those two dates and reset the row index

except Exception as e:
    print(f"Weather API failed: {e}")
    used_fallback = True

print(weather)
print("Used fallback:", used_fallback)

        time  temperature_2m_max  precipitation_sum
0 2026-09-05                29.4                2.5
1 2026-10-03                22.0               14.7
Used fallback: False


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [32]:
FALLBACK = pd.DataFrame({                   # Create sample weather data to use if the API request fails
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})
if used_fallback:                           # Check whether the API request failed
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weather.copy()
    wx['source'] = 'open-meteo'

print(wx)

        time  temperature_2m_max  precipitation_sum      source
0 2026-09-05                29.4                2.5  open-meteo
1 2026-10-03                22.0               14.7  open-meteo


# **Call did not fail, fallback not used**

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [33]:
wx = wx.rename(columns={                  # Rename the columns to use consistent names in your own data pipeline
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])                 # Convert the date column from strings to datetime values

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]    # Keep only the required columns in the specified order

display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [34]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [35]:
date1 = wx.loc[wx['date'] == pd.Timestamp('2026-09-05'), 'precip_mm'].iloc[0]   # Get the precipitation value for September 5, 2026
date2 = wx.loc[wx['date'] == pd.Timestamp('2026-10-03'), 'precip_mm'].iloc[0]   # Get the precipitation value for October 3, 2026

difference = abs(date2 - date1)       # Calculate difference in precipitation

if date2 > date1:
    print(f"October 3 was wetter than September 5 by {difference:.1f} mm of precipitation.")
elif date1 > date2:
    print(f"September 5 was wetter than October 3 by {difference:.1f} mm of precipitation.")
else:
    print("September 5 and October 3 had the same total precipitation.")

October 3 was wetter than September 5 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [36]:
from datetime import datetime
from zoneinfo import ZoneInfo

print("Weather Data Provenance")
print("-----------------------")
print(f"Endpoint: {BASE}")
print("Coordinates: latitude=38.03, longitude=-78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print(f"Pulled at: {datetime.now(ZoneInfo('America/New_York')).isoformat()}")
print(f"Data source: {wx['source'].iloc[0]}")
print(f"Fallback used: {used_fallback}")

Weather Data Provenance
-----------------------
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude=38.03, longitude=-78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-11T16:26:19.290856-04:00
Data source: open-meteo
Fallback used: False


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If `get_weather` returned an empty DataFrame on failure, merging it with 200 days of sales using a left join would preserve all sales rows but leave the weather columns missing. The chart might still display revenue, but rainfall would appear blank or incomplete, potentially suggesting there is no relationship between rain and revenue. Nobody might catch the problem because the merge and chart could run without errors, making the output appear valid even though the weather data was never retrieved. This is why the pipeline should validate that weather data exists and contains valid values before merging or plotting.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [37]:
charlottesville = get_weather('2026-09-05', '2026-10-03',lat=38.03, lon=-78.51)
orlando = get_weather('2026-09-05', '2026-10-03',lat=28.54, lon=-81.38)

target_dates = pd.to_datetime(['2026-09-05', '2026-10-03'])                             # Specify the two dates to keep

charlottesville['time'] = pd.to_datetime(charlottesville['time'])                       # Convert the time columns to datetime format
orlando['time'] = pd.to_datetime(orlando['time'])

charlottesville = charlottesville[charlottesville['time'].isin(target_dates)].copy()
orlando = orlando[orlando['time'].isin(target_dates)].copy()

charlottesville['city'] = 'Charlottesville'
orlando['city'] = 'Orlando'

wx_both = pd.concat([charlottesville, orlando],ignore_index=True)                    # Combine both cities' weather data into one DataFrame

wx_both = wx_both.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx_both['date'] = pd.to_datetime(wx_both['date'])

wx_both = wx_both[['city', 'date', 'temp_max', 'precip_mm']]

display(wx_both)

,city,date,temp_max,precip_mm
0,Charlottesville,2026-09-05,29.4,2.5
1,Charlottesville,2026-10-03,22.0,14.7
2,Orlando,2026-09-05,32.1,3.9
3,Orlando,2026-10-03,30.4,14.5
